In [1]:
import pydicom
import numpy as np
import cv2
import os
from PIL import Image
import pylibjpeg

input_root = "./data"
output_root = "./output_images"
output_format = "png"  # 또는 'jpg'

# 대상 폴더 이름 목록
target_folders = ["pre indx"]

for root, _, files in os.walk(input_root):
    if os.path.basename(root).lower() not in target_folders:
        continue

    for file in files:
        if file.lower().endswith(".dcm"):
            dicom_path = os.path.join(root, file)

            try:
                ds = pydicom.dcmread(dicom_path)
                pixel_array = ds.pixel_array.astype(np.float32)

                img_norm = cv2.normalize(pixel_array, None, 0, 255, cv2.NORM_MINMAX)
                img_8bit = np.uint8(img_norm)

                # 저장 경로 만들기
                relative_path = os.path.relpath(dicom_path, input_root)
                save_path = os.path.splitext(relative_path)[0] + f".{output_format}"
                full_save_path = os.path.join(output_root, save_path)

                os.makedirs(os.path.dirname(full_save_path), exist_ok=True)
                img_rgb = Image.fromarray(img_8bit)
                img_rgb.save(full_save_path)

                print(f"✅ Saved: {full_save_path}")
            except Exception as e:
                print(f"❌ Failed to convert {dicom_path} — {e}")


In [2]:
import pydicom
import numpy as np
import cv2
import os
from PIL import Image
import pylibjpeg

input_root = "./data"
output_root = "./output_images"
output_format = "png"  # 또는 'jpg'

target_folders = ["pre_indx"]

# (환자 번호: 변환할 인덱스 리스트)
target_dict = {
    4: [34],
    5: [42, 43, 44],
    9: [36],
    21: [43]
}

for root, _, files in os.walk(input_root):
    folder_name = os.path.basename(root)
    if folder_name.lower() not in target_folders:
        continue

    # 환자번호 추출 (폴더명 예: '4_00000000 홍길동/pre_indx')
    try:
        patient_base = root.split(os.sep)
        # 환자 폴더 찾기 ('4_206...' 형식)
        patient_id_str = [seg for seg in patient_base if '_' in seg and seg.split('_')[0].isdigit()][0]
        patient_id = int(patient_id_str.split('_')[0])
    except Exception:
        continue

    if patient_id not in target_dict:
        continue

    for file in files:
        if not file.lower().endswith(".dcm"):
            continue

        # 슬라이스 인덱스(예: '34-dicom-00..dcm' → 34 추출)
        slice_prefix = file.split('-')[0]
        if not (slice_prefix.isdigit() and int(slice_prefix) in target_dict[patient_id]):
            continue

        dicom_path = os.path.join(root, file)

        try:
            ds = pydicom.dcmread(dicom_path)
            pixel_array = ds.pixel_array.astype(np.float32)
            img_norm = cv2.normalize(pixel_array, None, 0, 255, cv2.NORM_MINMAX)
            img_8bit = np.uint8(img_norm)
            # 저장 경로 만들기
            relative_path = os.path.relpath(dicom_path, input_root)
            save_path = os.path.splitext(relative_path)[0] + f".{output_format}"
            full_save_path = os.path.join(output_root, save_path)
            os.makedirs(os.path.dirname(full_save_path), exist_ok=True)
            img_rgb = Image.fromarray(img_8bit)
            img_rgb.save(full_save_path)
            print(f"✅ Saved: {full_save_path}")
        except Exception as e:
            print(f"❌ Failed to convert {dicom_path} — {e}")


## 실행 검증용 데모 (합성 익명 데이터)

아래 셀은 위 변환 로직이 실제로 정상 동작함을 보이기 위한 것으로, **실제 환자 데이터가 아닌 무작위로 생성한 합성 DICOM**을 임시 폴더에 만들고 위와 동일한 방식으로 변환한 뒤 결과를 확인하고 삭제합니다. 실제 학습에는 원내(양산부산대학교병원) DICOM 데이터를 사용했으며, 그 데이터는 이 저장소에 포함하지 않습니다.

In [3]:
import os
import tempfile
import shutil
import numpy as np
import pydicom
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid


def _make_synthetic_dicom(path, rows=64, cols=64, instance_number=1, seed=0):
    """실행 검증용 합성 DICOM 1장 생성 (실제 환자 데이터 아님)."""
    rng = np.random.default_rng(seed)
    pixel_array = (rng.random((rows, cols)) * 200).astype(np.uint16)
    pixel_array[20:28, 20:28] += 400  # 시연용 밝은 영역

    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = pydicom.uid.MRImageStorage
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = FileDataset(path, {}, file_meta=file_meta, preamble=b"\0" * 128)
    ds.PatientName = "SYNTHETIC^DEMO"
    ds.PatientID = "SYNTH0000"
    ds.Modality = "MR"
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.InstanceNumber = instance_number
    ds.Rows, ds.Columns = rows, cols
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 0
    ds.PixelData = pixel_array.tobytes()
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ds.save_as(path, write_like_original=False)


demo_root = tempfile.mkdtemp(prefix="dicom2png_demo_")
try:
    # target_dict 규칙(환자번호_아무거나/pre_indx/{슬라이스번호}-...)에 맞춘 익명 합성 데이터
    _make_synthetic_dicom(os.path.join(demo_root, "4_00000000 홍길동", "pre_indx", "34-dicom-0001.dcm"), instance_number=1)
    _make_synthetic_dicom(os.path.join(demo_root, "5_00000001 김철수", "pre_indx", "42-dicom-0001.dcm"), instance_number=1, seed=1)
    _make_synthetic_dicom(os.path.join(demo_root, "5_00000001 김철수", "pre_indx", "43-dicom-0002.dcm"), instance_number=2, seed=2)

    demo_input_root = demo_root
    demo_output_root = os.path.join(demo_root, "output_images")

    for root, _, files in os.walk(demo_input_root):
        folder_name = os.path.basename(root)
        if folder_name.lower() not in target_folders:
            continue
        try:
            patient_base = root.split(os.sep)
            patient_id_str = [seg for seg in patient_base if '_' in seg and seg.split('_')[0].isdigit()][0]
            patient_id = int(patient_id_str.split('_')[0])
        except Exception:
            continue
        if patient_id not in target_dict:
            continue
        for file in files:
            if not file.lower().endswith(".dcm"):
                continue
            slice_prefix = file.split('-')[0]
            if not (slice_prefix.isdigit() and int(slice_prefix) in target_dict[patient_id]):
                continue
            dicom_path = os.path.join(root, file)
            try:
                ds = pydicom.dcmread(dicom_path)
                pixel_array = ds.pixel_array.astype(np.float32)
                img_norm = cv2.normalize(pixel_array, None, 0, 255, cv2.NORM_MINMAX)
                img_8bit = np.uint8(img_norm)
                relative_path = os.path.relpath(dicom_path, demo_input_root)
                save_path = os.path.splitext(relative_path)[0] + f".{output_format}"
                full_save_path = os.path.join(demo_output_root, save_path)
                os.makedirs(os.path.dirname(full_save_path), exist_ok=True)
                img_rgb = Image.fromarray(img_8bit)
                img_rgb.save(full_save_path)
                print(f"[DEMO] Saved: {os.path.relpath(full_save_path, demo_root)}")
            except Exception as e:
                print(f"[DEMO] Failed to convert {dicom_path} — {e}")

    saved_pngs = sorted(
        os.path.relpath(os.path.join(dp, f), demo_root)
        for dp, _, fs in os.walk(demo_output_root) for f in fs
    )
    print(f"\n총 {len(saved_pngs)}개 PNG 생성됨: {saved_pngs}")
    assert len(saved_pngs) == 3, "합성 데이터 변환 결과 개수가 예상과 다릅니다"
finally:
    shutil.rmtree(demo_root, ignore_errors=True)


[DEMO] Saved: output_images/4_00000000 홍길동/pre_indx/34-dicom-0001.png
[DEMO] Saved: output_images/5_00000001 김철수/pre_indx/42-dicom-0001.png
[DEMO] Saved: output_images/5_00000001 김철수/pre_indx/43-dicom-0002.png

총 3개 PNG 생성됨: ['output_images/4_00000000 홍길동/pre_indx/34-dicom-0001.png', 'output_images/5_00000001 김철수/pre_indx/42-dicom-0001.png', 'output_images/5_00000001 김철수/pre_indx/43-dicom-0002.png']
